Make sure the right schema is used

In [0]:
USE CATALOG sac;

USE SCHEMA customer_service;

In [0]:
SELECT
	zone,
	count(*)
FROM
	customer
GROUP BY
	zone

zone,count(1)
Niedersachsen,2
Thüringen,6229
Schleswig-Holstein,11080
Sachsen,12023
Hamburg,3351
Berlin,4774
null,117
Nordrhein-Westfalen,52168
Brandenburg,10255


# Gold Tables
average customer

In [0]:
CREATE OR REPLACE VIEW average_customer AS
SELECT
	zone,
	count(*) AS amount_customers,
	round(avg(monthly_bill), 2) AS avg_monthly_bill,
	round(avg(speed_tier_mbps), 0) AS avg_speed_tier,
	round(avg(data_usage_gb_last_month), 2) AS avg_data_usage
FROM
	customer
GROUP BY
	zone;

tickets per customer

In [0]:
CREATE OR REPLACE VIEW customer_connection_ticket_count AS
SELECT
	c.customer_id,
	count(DISTINCT l.timestamp) AS connection_fails,
	count(DISTINCT s.ticket_id) AS tickets,
	count(DISTINCT ca.session_id) AS chats,
	ch.churned AS churned
FROM
	customer c
		JOIN ticket s
			ON c.customer_id = s.customer_id
		LEFT JOIN log l
			ON c.customer_id = l.customer_id
			AND l.issue_detected != 'none'
		LEFT JOIN churn ch
			ON c.customer_id = ch.customer_id
		LEFT JOIN chat ca
			ON c.customer_id = ca.customer_id
GROUP BY
	c.customer_id,
	ch.churned;

churned customer

In [0]:
CREATE OR REPLACE VIEW churned_customer_details AS
SELECT
	c.customer_id,
	c.speed_tier_mbps,
	c.monthly_bill,
	count(DISTINCT l.timestamp) AS connection_fails,
	count(DISTINCT s.ticket_id) AS tickets,
	count(DISTINCT ca.session_id) AS chats
FROM
	customer c
		LEFT JOIN ticket s
			ON c.customer_id = s.customer_id
		LEFT JOIN log l
			ON c.customer_id = l.customer_id
			AND l.issue_detected != 'none'
		LEFT JOIN chat ca
			ON c.customer_id = ca.customer_id
		JOIN churn ch
			ON c.customer_id = ch.customer_id
WHERE
	ch.churned = TRUE
GROUP BY
	c.customer_id,
	c.speed_tier_mbps,
	c.monthly_bill;

location detail

In [0]:
CREATE OR REPLACE VIEW location_detail AS
WITH revenue_per_location AS (
	SELECT
		zone,
		sum(monthly_bill) AS revenue
	FROM
		customer
	GROUP BY
		zone
),
issues_per_location AS (
	SELECT
		c.zone,
		count(
			CASE
				WHEN l.issue_detected != 'none' THEN 1
			END
		) AS issue_count
	FROM
		customer c
			LEFT JOIN log l
				ON c.customer_id = l.customer_id
	GROUP BY
		c.zone
)
SELECT
	c.zone,
	count(DISTINCT c.customer_id) AS customer_count,
	round(r.revenue / 1000, 2) AS revenue_in_t,
	i.issue_count AS issue_count,
	count(DISTINCT t.ticket_id) AS ticket_count,
	count(DISTINCT ch.session_id) AS chat_count
FROM
	customer c
		LEFT JOIN revenue_per_location r
			ON c.zone = r.zone
		LEFT JOIN issues_per_location i
			ON c.zone = i.zone
		LEFT JOIN ticket t
			ON c.customer_id = t.customer_id
		LEFT JOIN chat ch
			ON c.customer_id = ch.customer_id
GROUP BY
	c.zone,
	r.revenue,
	i.issue_count;

average connection quality

In [0]:
CREATE OR REPLACE VIEW average_connection_quality AS
SELECT
	c.zone,
	round(avg(l.speed_measured_mbps), 0) AS avg_speed,
	round(avg(l.packet_loss_percent), 2) AS avg_packet_loss,
	round(avg(l.latency_ms), 2) AS avg_latency,
	round(avg(l.downtime_minutes), 2) AS avg_downtime,
	round(avg(l.connection_drops_count), 2) AS avg_connection_drops,
	count(
		CASE
			WHEN l.issue_detected != 'none' THEN 1
			ELSE 0
		END
	) AS count_issues
FROM
	log l
		LEFT JOIN customer c
			ON l.customer_id = c.customer_id
GROUP BY
	zone;

chat issues

In [0]:
CREATE OR REPLACE VIEW chat_issues AS
SELECT
	c.classification,
	m.sentiment,
	count(m.sentiment) AS count,
	FIRST(c.comment) AS exmp_comment
FROM
	chat c LEFT JOIN message m
WHERE
	classification IS NOT NULL
	AND m.speaker = 'customer'
GROUP BY
	c.classification,
	m.sentiment
ORDER BY
	count DESC

sentiment for agent

In [0]:
CREATE OR REPLACE VIEW sentiment_for_agent AS
SELECT
	concat(a.first_name, ' ', a.last_name) AS agent_name,
	m.sentiment,
	count(m.sentiment) AS count
FROM
	chat c
		JOIN message m
			ON c.session_id = m.session_id
			AND m.speaker = 'customer'
		LEFT JOIN agent a
			ON c.agent_id = a.agent_id
WHERE
	a.last_name IS NOT NULL
GROUP BY
	agent_name,
	m.sentiment;

# Show tables

In [0]:
SELECT
	*
FROM
	average_customer;

zone,amount_customers,avg_monthly_bill,avg_speed_tier,avg_data_usage
Niedersachsen,2,47.49,50.0,195.85
Thüringen,5873,75.33,298.0,305.5
Schleswig-Holstein,11100,75.92,306.0,294.5
Sachsen,11919,75.79,305.0,298.99
Hamburg,3351,76.19,308.0,307.47
Berlin,4771,75.48,301.0,293.72
null,508,75.8,300.0,303.22
Nordrhein-Westfalen,52201,75.94,308.0,300.97
Brandenburg,10195,75.87,308.0,298.33
Bayern,6,74.66,283.0,136.53


In [0]:
SELECT
	*
FROM
	customer_connection_ticket_count
ORDER BY
	tickets DESC
LIMIT 20;

customer_id,connection_fails,tickets,chats,churned
CUST_00971,9,3,0,true
CUST_01051,5,3,0,false
CUST_01086,4,3,0,false
CUST_00739,13,3,2,false
CUST_01021,5,3,0,false
CUST_00865,6,3,2,false
CUST_00003,8,2,0,true
CUST_00460,5,2,2,false
CUST_00761,3,2,0,false
CUST_00133,4,2,1,false


In [0]:
SELECT
	*
FROM
	churned_customer_details
ORDER BY
	tickets DESC
LIMIT 20;

customer_id,speed_tier_mbps,monthly_bill,connection_fails,tickets,chats
CUST_00971,200,79.99,9,3,0
CUST_00577,50,44.991,2,2,2
CUST_00106,50,49.99,10,2,1
CUST_00695,50,49.99,4,2,0
CUST_00284,50,49.99,4,2,0
CUST_00003,200,71.991,8,2,0
CUST_00615,1000,119.99,14,2,0
CUST_00937,50,49.99,8,2,2
CUST_00021,200,71.991,5,2,0
CUST_00508,1000,119.99,2,2,1


In [0]:
SELECT
	*
FROM
	location_detail
ORDER BY
	customer_count DESC;

zone,customer_count,revenue_in_t,issue_count,ticket_count,chat_count
Nordrhein-Westfalen,52201,3964.27,32184,0,0
Sachsen,11919,903.3,7354,0,0
Schleswig-Holstein,11100,842.67,6843,0,0
Brandenburg,10195,773.46,6039,0,0
Thüringen,5873,442.39,3590,0,0
Berlin,4771,360.14,2686,0,0
Hamburg,3351,255.3,2027,0,0
null,508,null,null,0,0
Sachsen-Anhalt,51,3.9,17,0,0
Baden-Württemberg,9,0.58,0,0,0


In [0]:
SELECT
	*
FROM
	average_connection_quality;

zone,avg_speed,avg_packet_loss,avg_latency,avg_downtime,avg_connection_drops,count_issues
Niedersachsen,41.0,5.27,44.6,10.0,3.96,28
Thüringen,233.0,3.41,47.31,3.36,2.66,20159
Bayern,44.0,3.25,69.71,3.13,3.0,47
Schleswig-Holstein,260.0,3.25,47.24,2.9,2.4,38457
Hamburg,271.0,3.26,47.78,2.93,2.56,11473
Sachsen,254.0,3.43,47.16,3.23,2.6,40778
Berlin,257.0,3.38,47.4,3.15,2.67,15039
null,316.0,3.54,47.51,3.83,2.86,1757
Sachsen-Anhalt,163.0,1.42,42.3,0.0,0.13,165
Nordrhein-Westfalen,268.0,3.41,47.37,3.22,2.59,179032


In [0]:
SELECT
	*
FROM
	chat_issues
ORDER BY
	classification,
	sentiment;

classification,sentiment,count,exmp_comment
OTHER,Negativ,30334,Bietet Unterstützung an und fordert zur Kontaktaufnahme auf bei weiteren Problemen.
OTHER,Neutral,53650,Überweisung erfolgt innerhalb von 3-5 Werktagen.
OTHER,Positiv,6612,Bietet Unterstützung an und fordert zur Kontaktaufnahme bei Problemen auf.
OTHER,Unbekannt,1798,Überweisung erfolgt innerhalb von 3-5 Werktagen.
PRICE,Negativ,19874,Rechnungen belaufen sich auf 40 Euro monatlich
PRICE,Neutral,35150,15 Euro werden automatisch als Guthaben abgezogen
PRICE,Positiv,4332,15 Euro werden automatisch als Guthaben abgezogen
PRICE,Unbekannt,1178,Rechnungen belaufen sich auf 40 Euro monatlich
SERVICE,Negativ,205539,Technikertermin wird vereinbart
SERVICE,Neutral,363525,Link zu WLAN-Optimierungsartikel wird per E-Mail gesendet


In [0]:
SELECT
	*
FROM
	sentiment_for_agent
ORDER BY
	agent_name,
	sentiment
LIMIT 20;

agent_name,sentiment,count
null,Negativ,26
null,Neutral,50
null,Positiv,7
null,Unbekannt,2
Anna Schmidt,Negativ,27
Anna Schmidt,Neutral,47
Anna Schmidt,Positiv,4
Ben Neumann,Negativ,24
Ben Neumann,Neutral,43
Ben Neumann,Positiv,3
